# GraphForge — Rejection-Sampling SFT on a free Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This notebook runs the full pipeline end-to-end against the GraphForge OpenEnv environment:

1. Clone the repo and install deps (1 min on T4)
2. Baseline-eval Qwen2.5-0.5B-Instruct against the tier-0 task
3. Generate trajectories (oracle + live model) and reject-sample
4. SFT the kept trajectories (TRL SFTTrainer + LoRA)
5. Trained-eval the same model and write all hackathon plots

Expected wall-clock: ~10–20 min on a T4.

## 1. Setup

In [ ]:
# Clone the GraphForge repo. Replace this URL with your fork once you've published it.
REPO_URL = 'https://github.com/<your-username>/graphforge.git'

import os, subprocess, sys, pathlib
if not pathlib.Path('graphforge').exists():
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, 'graphforge_repo'])
    os.chdir('graphforge_repo')
else:
    os.chdir('graphforge_repo')
print('cwd:', os.getcwd())
print(os.listdir('.'))

In [ ]:
# Install runtime + training deps. peft & trl handle the SFT side.
%pip install -q -e ".[training]"
%pip install -q peft
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Run the full pipeline

`training.train.run` does baseline eval → trajectory generation → SFT → trained eval → plots, all in one call.

In [ ]:
from pathlib import Path
from training.config import TrainConfig
from training.train import run

cfg = TrainConfig(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    task_id='t0.email_validator',
    n_oracle=20,
    n_explore=30,
    reward_threshold=5.0,
    epochs=2,
    learning_rate=1e-4,
    batch_size=1,
    gradient_accumulation_steps=4,
    use_lora=True,
    n_eval_episodes=20,
    out_dir=Path('outputs'),
    plots_dir=Path('plots'),
)
summary = run(cfg)
summary['baseline_eval']['mean_reward'], summary['trained_eval']['mean_reward']

## 3. Show the plots

In [ ]:
from IPython.display import Image, display
for name in ['comparison.png', 'baseline_rewards.png', 'trained_rewards.png', 'loss_curve.png']:
    p = Path('plots') / name
    if p.exists():
        print(name)
        display(Image(str(p)))

## 4. Commit the plots back to the repo

Once you're happy with the run, copy `plots/*.png` and `outputs/summary.json` into your fork and push. The README embeds them automatically.